In [2]:
import numpy as np

In [6]:
from data_loader import load_mnist

x_train, y_train, x_test, y_test = load_mnist()

In [9]:
print(f"Training set shape: {x_train.shape}, {y_train.shape}")
x_train_split, x_val = x_train[:50000], x_train[50000:]
y_train_split, y_val = y_train[:50000], y_train[50000:]
print(f"Training split shape: {x_train_split.shape}, {y_train_split.shape}")
print(f"Validation set shape: {x_val.shape}, {y_val.shape}")

y_train_split = (y_train_split == 5).astype(float).reshape(-1, 1)
y_val = (y_val == 5).astype(float).reshape(-1, 1)
print(f"y_train_split shape: {y_train_split.shape}")
print(f"y_val shape: {y_val.shape}")

Training set shape: (60000, 28, 28), (60000,)
Training split shape: (50000, 28, 28), (50000,)
Validation set shape: (10000, 28, 28), (10000,)
y_train_split shape: (50000, 1)
y_val shape: (10000, 1)


In [18]:
x_train_split, x_val_split = x_train[:50000], x_train[50000:]
y_train_split, y_val_split = y_train[:50000], y_train[50000:]
    
x_train_split = x_train_split / 255.0
x_val_split = x_val_split / 255.0

y_c1_c2_train = (y_train_split == 5) | (y_train_split == 6)  # (split,) bool
y_c1_c2_val   = (y_val_split   == 5) | (y_val_split   == 6)  # (split',) bool

    # Filter x to only c1/c2 samples
x_train_filtered = x_train_split[y_c1_c2_train]  # (n_train, 28, 28)
x_val_filtered   = x_val_split[y_c1_c2_val]      # (n_val,   28, 28)

    # Single binary label: 1 for c1, 0 for c2
y_train_split = (y_train_split[y_c1_c2_train] == 5).astype(float).reshape(-1, 1)  # (n_train, 1)
y_val   = (y_val_split  [y_c1_c2_val]   == 5).astype(float).reshape(-1, 1)  # (n_val,   1)

x_train_flat = x_train_filtered.reshape(x_train_filtered.shape[0], -1) # (split, 784)
x_train_bias = np.hstack([x_train_flat, np.ones((x_train_filtered.shape[0], 1))])  # (50000, 785, 1)

x_val_flat = x_val_filtered.reshape(x_val_filtered.shape[0], -1)  # (split', 785)
x_val_bias = np.hstack([x_val_flat, np.ones((x_val_filtered.shape[0], 1))]) # (split', 785)

print(f"x_train_bias shape: {x_train_bias.shape}")
print(f"x_val_bias shape: {x_val_bias.shape}")
print(f"y_train_split shape: {y_train_split.shape}")
print(f"y_val shape: {y_val.shape}")

x_train_bias shape: (9457, 785)
x_val_bias shape: (1882, 785)
y_train_split shape: (9457, 1)
y_val shape: (1882, 1)


In [19]:
def logistic_regression(x_train, y_train, model_digit_c1, model_digit_c2, batch_size=32, learning_rate=0.01, validation_patience=5, epsilon=1e-4):
    
    x_train_split, x_val_split = x_train[:50000], x_train[50000:]
    y_train_split, y_val_split = y_train[:50000], y_train[50000:]
    
    x_train_split = x_train_split / 255.0
    x_val_split = x_val_split / 255.0

    y_c1_c2_train = (y_train_split == model_digit_c1) | (y_train_split == model_digit_c2)  # (split,) bool
    y_c1_c2_val   = (y_val_split   == model_digit_c1) | (y_val_split   == model_digit_c2)  # (split',) bool

    # Filter x to only c1/c2 samples
    x_train_filtered = x_train_split[y_c1_c2_train]  # (n_train, 28, 28)
    x_val_filtered   = x_val_split[y_c1_c2_val]      # (n_val,   28, 28)

    # Single binary label: 1 for c1, 0 for c2
    y_train_split = (y_train_split[y_c1_c2_train] == model_digit_c1).astype(float).reshape(-1, 1)  # (n_train, 1)
    y_val   = (y_val_split  [y_c1_c2_val]   == model_digit_c1).astype(float).reshape(-1, 1)  # (n_val,   1)

    x_train_flat = x_train_filtered.reshape(x_train_filtered.shape[0], -1) # (split, 784)
    x_train_bias = np.hstack([x_train_flat, np.ones((x_train_filtered.shape[0], 1))])  # (50000, 785, 1)

    x_val_flat = x_val_filtered.reshape(x_val_filtered.shape[0], -1)  # (split', 785)
    x_val_bias = np.hstack([x_val_flat, np.ones((x_val_filtered.shape[0], 1))]) # (split', 785)

    w = np.random.uniform(low=-0.01, high=0.01, size=(785, 1))
    best_v_loss = float('inf')
    best_w = w.copy()
    patience = validation_patience
    counter = 0
    epoch = 0

    N = x_train_bias.shape[0]
    n_pos = np.sum(y_train_split == 1)
    n_neg = N - n_pos

    # Note dividing by 2 is esential to keep the overall average weight at 1
    # hence keeping the same learning rate scale as if we were doing unweighted logistic regression
    w_pos = N / (2 * n_pos)
    w_neg = N / (2 * n_neg)

    max_epochs = 100
    while epoch < max_epochs:
        #Since we are doing Mini-batch gradient descent, 
        # we need to shuffle the data to prevent the model from learning the order of the data.
        indices = np.random.permutation(N)
        x_shuffled = x_train_bias[indices]
        y_shuffled = y_train_split[indices]

        for start in range(0, N, batch_size):

            end   = min(start + batch_size, N)
            
            xi_batch = x_shuffled[start:end]          # (B, 785)
            yi_batch = y_shuffled[start:end]                      # (B, 1)

            z_batch  = np.dot(xi_batch, w)              # (B, 1)
            y_hat    = 1 / (1 + np.exp(-z_batch))       # (B, 1)

            sample_weights = yi_batch * w_pos + (1 - yi_batch) * w_neg
            error   = (y_hat - y_shuffled[start:end]) * sample_weights   # (B, 1)
            de_dw   = np.dot(xi_batch.T, error) / xi_batch.shape[0]  # (785, 1)
            w      -= learning_rate * de_dw
            
        z_v = np.dot(x_val_bias, w) #split'x785 * 785x1
        y_hat_v = 1 / (1 + np.exp(-z_v))

        y_hat_v = np.clip(y_hat_v, 1e-15, 1 - 1e-15) # Cap to avoid log(0) errors

        v_loss = -np.mean(y_val * np.log(y_hat_v) + (1 - y_val) * np.log(1 - y_hat_v))

        if v_loss < best_v_loss - epsilon:
            best_v_loss = v_loss
            best_w = w.copy()
            counter = 0
        else:
            counter += 1

        if counter == patience:
            return best_w
        print(f"Epoch: {epoch}, Loss:  {v_loss}")

        epoch += 1

        if(epoch == max_epochs):
            print("Reached maximum epochs. Stopping training...")
            return best_w


In [20]:
from itertools import combinations

weights_dict = {}

digits = list(range(10))  # 0 → 9

for c1, c2 in combinations(digits, 2):
    print(f"Training model for {c1} vs {c2}")
    
    w = logistic_regression(
        x_train, y_train,
        c1, c2,
        batch_size=1,
        learning_rate=0.01,
        validation_patience=5,
        epsilon=1e-4
    )
    
    weights_dict[(c1, c2)] = w

Training model for 0 vs 1
Epoch: 0, Loss:  0.005082429044199909
Epoch: 1, Loss:  0.0034572784433318116
Epoch: 2, Loss:  0.0027599845881689246
Epoch: 3, Loss:  0.0025584767325116517
Epoch: 4, Loss:  0.0027881993575987387
Epoch: 5, Loss:  0.0023106295258338255
Epoch: 6, Loss:  0.0019123723871826272
Epoch: 7, Loss:  0.0016419793098248656
Epoch: 8, Loss:  0.0017602921054422661
Epoch: 9, Loss:  0.0017820234500139792
Epoch: 10, Loss:  0.0017758068279869425
Epoch: 11, Loss:  0.0015035248917863364
Epoch: 12, Loss:  0.0013783760634965279
Epoch: 13, Loss:  0.0013869276284022567
Epoch: 14, Loss:  0.001325159701681881
Epoch: 15, Loss:  0.0013760662540976906
Epoch: 16, Loss:  0.0014009728972986361
Training model for 0 vs 2
Epoch: 0, Loss:  0.03357189343678377
Epoch: 1, Loss:  0.027959654637883986
Epoch: 2, Loss:  0.024323011249418634
Epoch: 3, Loss:  0.024918125683085896
Epoch: 4, Loss:  0.02324926818325399
Epoch: 5, Loss:  0.023736139299501278
Epoch: 6, Loss:  0.02724632458419143
Epoch: 7, Loss:  

In [21]:
def evaluate_digit_ovo(weights_dict, x_test, y_test, target_digit):
    
    # ---- Step 1: Predict all ----
    def predict_one(x):
        votes = {i: 0 for i in range(10)}

        x_flat = x.reshape(1, -1) / 255.0
        x_bias = np.hstack([x_flat, np.ones((1, 1))])

        for (c1, c2), w in weights_dict.items():
            z = np.dot(x_bias, w)
            y_hat = 1 / (1 + np.exp(-z))

            if y_hat >= 0.5:
                votes[c1] += 1
            else:
                votes[c2] += 1

        return max(votes, key=votes.get)

    y_pred = np.array([predict_one(x) for x in x_test])

    # ---- Step 2: Convert to binary (target vs all) ----
    y_true_bin = (y_test == target_digit).astype(int)
    y_pred_bin = (y_pred == target_digit).astype(int)

    # ---- Step 3: Compute metrics ----
    tp = np.sum((y_pred_bin == 1) & (y_true_bin == 1))
    tn = np.sum((y_pred_bin == 0) & (y_true_bin == 0))
    fp = np.sum((y_pred_bin == 1) & (y_true_bin == 0))
    fn = np.sum((y_pred_bin == 0) & (y_true_bin == 1))

    accuracy  = (tp + tn) / len(y_test)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    # ---- Step 4: Print ----
    print(f"Results for digit {target_digit} vs All")
    print(f"Accuracy  : {accuracy * 100:.2f}%")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")

evaluate_digit_ovo(weights_dict, x_test, y_test, 8)

Results for digit 8 vs All
Accuracy  : 98.34%
Precision : 0.9253
Recall    : 0.9025
F1 Score  : 0.9137
TP: 879, TN: 8955, FP: 71, FN: 95


In [23]:
def logistic_regression_k_folds(x_train, y_train, model_digit_c1, model_digit_c2,
                             k=5, batch_size=32, learning_rate=0.01,
                             validation_patience=5, epsilon=1e-4):

    # --- 1. Normalize and filter to the two classes ONCE up front ---
    x = x_train / 255.0

    mask = (y_train == model_digit_c1) | (y_train == model_digit_c2)
    x = x[mask]
    y = (y_train[mask] == model_digit_c1).astype(float).reshape(-1, 1)

    N = x.shape[0]

    # --- 2. Shuffle once before folding ---
    indices = np.random.permutation(N)
    x = x[indices]
    y = y[indices]

    # --- 3. Split into k roughly-equal folds ---
    fold_indices = np.array_split(np.arange(N), k)

    fold_weights = []
    fold_val_losses = []

    for fold_idx in range(k):
        print(f"\n{'='*40}")
        print(f"Fold {fold_idx + 1} / {k}")
        print(f"{'='*40}")

        # --- 4. Build train / val splits for this fold ---
        val_idx   = fold_indices[fold_idx]
        train_idx = np.concatenate([fold_indices[i] for i in range(k) if i != fold_idx])

        x_val_fold   = x[val_idx]
        y_val_fold   = y[val_idx]
        x_train_fold = x[train_idx]
        y_train_fold = y[train_idx]

        # --- 5. Flatten + add bias column ---
        x_train_flat = x_train_fold.reshape(x_train_fold.shape[0], -1)
        x_train_bias = np.hstack([x_train_flat, np.ones((x_train_flat.shape[0], 1))])

        x_val_flat   = x_val_fold.reshape(x_val_fold.shape[0], -1)
        x_val_bias   = np.hstack([x_val_flat, np.ones((x_val_flat.shape[0], 1))])

        # --- 6. Class weights (same logic as original) ---
        n_fold   = x_train_bias.shape[0]
        n_pos    = np.sum(y_train_fold == 1)
        n_neg    = n_fold - n_pos
        w_pos    = n_fold / (2 * n_pos)
        w_neg    = n_fold / (2 * n_neg)

        # --- 7. Train using the same loop as your original function ---
        w = np.random.uniform(low=-0.01, high=0.01, size=(785, 1))
        best_val_loss = float('inf')
        best_w  = w.copy()
        counter = 0
        epoch   = 0
        max_epochs = 100

        while epoch < max_epochs:
            perm = np.random.permutation(n_fold)
            x_shuf = x_train_bias[perm]
            y_shuf = y_train_fold[perm]

            for start in range(0, n_fold, batch_size):
                end      = min(start + batch_size, n_fold)
                xi_batch = x_shuf[start:end]
                yi_batch = y_shuf[start:end]

                z        = np.dot(xi_batch, w)
                y_hat    = 1 / (1 + np.exp(-z))

                sample_weights = yi_batch * w_pos + (1 - yi_batch) * w_neg
                error    = (y_hat - yi_batch) * sample_weights
                de_dw    = np.dot(xi_batch.T, error) / xi_batch.shape[0]
                w       -= learning_rate * de_dw

            # Validation loss for early stopping
            z_v      = np.dot(x_val_bias, w)
            y_hat_v  = np.clip(1 / (1 + np.exp(-z_v)), 1e-15, 1 - 1e-15)
            v_loss   = -np.mean(y_val_fold * np.log(y_hat_v) +
                                (1 - y_val_fold) * np.log(1 - y_hat_v))

            if v_loss < best_val_loss - epsilon:
                best_val_loss = v_loss
                best_w  = w.copy()
                counter = 0
            else:
                counter += 1

            print(f"Epoch {epoch:3d} | val_loss: {v_loss:.5f}")

            if counter == validation_patience:
                print(f"Early stopping at epoch {epoch}")
                break

            epoch += 1
            if epoch == max_epochs:
                print("Reached max epochs.")

In [24]:
from itertools import combinations

weights_dict = {}

digits = list(range(10))  # 0 → 9

for c1, c2 in combinations(digits, 2):
    print(f"Training model for {c1} vs {c2}")
    
    w = logistic_regression_k_folds(
        x_train, y_train,
        c1, c2,
        k=5,
        batch_size=1,
        learning_rate=0.01,
        validation_patience=5,
        epsilon=1e-4
    )
    
    weights_dict[(c1, c2)] = w

Training model for 0 vs 1

Fold 1 / 5
Epoch   0 | val_loss: 0.00813
Epoch   1 | val_loss: 0.00708
Epoch   2 | val_loss: 0.00709
Epoch   3 | val_loss: 0.00691
Epoch   4 | val_loss: 0.00677
Epoch   5 | val_loss: 0.00682
Epoch   6 | val_loss: 0.00682
Epoch   7 | val_loss: 0.00684
Epoch   8 | val_loss: 0.00707
Epoch   9 | val_loss: 0.00690
Early stopping at epoch 9

Fold 2 / 5
Epoch   0 | val_loss: 0.00655
Epoch   1 | val_loss: 0.00429
Epoch   2 | val_loss: 0.00360
Epoch   3 | val_loss: 0.00310
Epoch   4 | val_loss: 0.00288
Epoch   5 | val_loss: 0.00267
Epoch   6 | val_loss: 0.00271
Epoch   7 | val_loss: 0.00251
Epoch   8 | val_loss: 0.00239
Epoch   9 | val_loss: 0.00241
Epoch  10 | val_loss: 0.00252
Epoch  11 | val_loss: 0.00234
Epoch  12 | val_loss: 0.00235
Epoch  13 | val_loss: 0.00227
Epoch  14 | val_loss: 0.00212
Epoch  15 | val_loss: 0.00216
Epoch  16 | val_loss: 0.00221
Epoch  17 | val_loss: 0.00205
Epoch  18 | val_loss: 0.00209
Epoch  19 | val_loss: 0.00209
Early stopping at epoch 

In [ ]:
def evaluate_digit_ovo(weights_dict, x_test, y_test, target_digit):
    
    # ---- Step 1: Predict all ----
    def predict_one(x):
        votes = {i: 0 for i in range(10)}

        x_flat = x.reshape(1, -1) / 255.0
        x_bias = np.hstack([x_flat, np.ones((1, 1))])

        for (c1, c2), w in weights_dict.items():
            z = np.dot(x_bias, w)
            y_hat = 1 / (1 + np.exp(-z))

            if y_hat >= 0.5:
                votes[c1] += 1
            else:
                votes[c2] += 1

        return max(votes, key=votes.get)

    y_pred = np.array([predict_one(x) for x in x_test])

    # ---- Step 2: Convert to binary (target vs all) ----
    y_true_bin = (y_test == target_digit).astype(int)
    y_pred_bin = (y_pred == target_digit).astype(int)

    # ---- Step 3: Compute metrics ----
    tp = np.sum((y_pred_bin == 1) & (y_true_bin == 1))
    tn = np.sum((y_pred_bin == 0) & (y_true_bin == 0))
    fp = np.sum((y_pred_bin == 1) & (y_true_bin == 0))
    fn = np.sum((y_pred_bin == 0) & (y_true_bin == 1))

    accuracy  = (tp + tn) / len(y_test)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    # ---- Step 4: Print ----
    print(f"Results for digit {target_digit} vs All")
    print(f"Accuracy  : {accuracy * 100:.2f}%")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")

def logistic_regression_k_folds(x_train, y_train, model_digit_c1, model_digit_c2,
                             k=5, batch_size=32, learning_rate=0.01,
                             validation_patience=5, epsilon=1e-4):

    # --- 1. Normalize and filter to the two classes ONCE up front ---
    x = x_train / 255.0

    mask = (y_train == model_digit_c1) | (y_train == model_digit_c2)
    x = x[mask]
    y = (y_train[mask] == model_digit_c1).astype(float).reshape(-1, 1)

    N = x.shape[0]

    # --- 2. Shuffle once before folding ---
    indices = np.random.permutation(N)
    x = x[indices]
    y = y[indices]

    # --- 3. Split into k roughly-equal folds ---
    fold_indices = np.array_split(np.arange(N), k)

    fold_weights = []
    fold_val_losses = []

    for fold_idx in range(k):
        print(f"\n{'='*40}")
        print(f"Fold {fold_idx + 1} / {k}")
        print(f"{'='*40}")

        # --- 4. Build train / val splits for this fold ---
        val_idx   = fold_indices[fold_idx]
        train_idx = np.concatenate([fold_indices[i] for i in range(k) if i != fold_idx])

        x_val_fold   = x[val_idx]
        y_val_fold   = y[val_idx]
        x_train_fold = x[train_idx]
        y_train_fold = y[train_idx]

        # --- 5. Flatten + add bias column ---
        x_train_flat = x_train_fold.reshape(x_train_fold.shape[0], -1)
        x_train_bias = np.hstack([x_train_flat, np.ones((x_train_flat.shape[0], 1))])

        x_val_flat   = x_val_fold.reshape(x_val_fold.shape[0], -1)
        x_val_bias   = np.hstack([x_val_flat, np.ones((x_val_flat.shape[0], 1))])

        # --- 6. Class weights (same logic as original) ---
        n_fold   = x_train_bias.shape[0]
        n_pos    = np.sum(y_train_fold == 1)
        n_neg    = n_fold - n_pos
        w_pos    = n_fold / (2 * n_pos)
        w_neg    = n_fold / (2 * n_neg)

        # --- 7. Train using the same loop as your original function ---
        w = np.random.uniform(low=-0.01, high=0.01, size=(785, 1))
        best_val_loss = float('inf')
        best_w  = w.copy()
        counter = 0
        epoch   = 0
        max_epochs = 100

        while epoch < max_epochs:
            perm = np.random.permutation(n_fold)
            x_shuf = x_train_bias[perm]
            y_shuf = y_train_fold[perm]

            for start in range(0, n_fold, batch_size):
                end      = min(start + batch_size, n_fold)
                xi_batch = x_shuf[start:end]
                yi_batch = y_shuf[start:end]

                z        = np.dot(xi_batch, w)
                y_hat    = 1 / (1 + np.exp(-z))

                sample_weights = yi_batch * w_pos + (1 - yi_batch) * w_neg
                error    = (y_hat - yi_batch) * sample_weights
                de_dw    = np.dot(xi_batch.T, error) / xi_batch.shape[0]
                w       -= learning_rate * de_dw

            # Validation loss for early stopping
            z_v      = np.dot(x_val_bias, w)
            y_hat_v  = np.clip(1 / (1 + np.exp(-z_v)), 1e-15, 1 - 1e-15)
            v_loss   = -np.mean(y_val_fold * np.log(y_hat_v) +
                                (1 - y_val_fold) * np.log(1 - y_hat_v))

            if v_loss < best_val_loss - epsilon:
                best_val_loss = v_loss
                best_w  = w.copy()
                counter = 0
            else:
                counter += 1

            print(f"Epoch {epoch:3d} | val_loss: {v_loss:.5f}")

            if counter == validation_patience:
                print(f"Early stopping at epoch {epoch}")
                break

            epoch += 1
            if epoch == max_epochs:
                print("Reached max epochs.")
        
        fold_weights.append(best_w)
        fold_val_losses.append(best_val_loss)
    
    # Average weights across all folds
    avg_w = np.mean(fold_weights, axis=0)
    return avg_w

TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'